# DOMCS-EEG Colab Follow-up Security Evaluation (Low-RAM)

This notebook is the low-RAM version for standard Colab GPU sessions.

Key idea:
- do **not** load the full `EEGMMIDB_win2s_step1s_fs128.npz` into RAM
- extract the `.npz` once into individual `.npy` files
- memory-map only the large `X` array
- load labels and run metadata normally
- reuse the shipped checkpoint if available

This is the recommended notebook if your Colab session crashes on the standard notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install numpy pandas scipy scikit-learn matplotlib torch

In [ ]:
import os
import sys
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import roc_curve, auc as sk_auc

REPO_ROOT = '/content/drive/MyDrive/github repo 12apr26/DOMCS_EEG_GITHUB_FINAL_20260406_2305/DOMCS-EEG'
NPZ_PATH = '/content/drive/MyDrive/EEG_Q1_BIOMETRIC_PROJECT/02_PREPROCESSING/outputs/EEGMMIDB_win2s_step1s_fs128.npz'
EXTRACT_DIR = '/content/drive/MyDrive/EEG_Q1_BIOMETRIC_PROJECT/02_PREPROCESSING/extracted_eegmmidb_fs128'
OUT_ROOT = '/content/drive/MyDrive/EEG_Q1_BIOMETRIC_PROJECT/03_ATTACK_RESULTS_DOMCS_LOW_RAM'

RUN_SMOKE_TEST = True
MAX_SUBJECTS_SMOKE = 10
MAX_PROBES_SMOKE = 3000

# For paper-final runs on standard Colab RAM:
# 1. Set RUN_SMOKE_TEST = False
# 2. Keep the chunk sizes below conservative limits
EMBED_BATCH_SIZE = 128
SCORE_CHUNK_SIZE = 4096
NOISE_CHUNK_SIZE = 2048
TRAIN_RUNS = ['r01', 'r02']
TEST_RUNS = ['r03', 'r04', 'r05', 'r06', 'r07', 'r08', 'r09', 'r10', 'r11', 'r12', 'r13', 'r14']
K_PROTOTYPES = 3
ATTACK_TARGET_SUBJECT_LIMIT = 10
ATTACK_IMPOSTOR_LIMIT = 10
ATTACK_WINDOWS_PER_IMPOSTOR = 32
ATTACK_BATCH_SIZE = 32
FGSM_EPS_LIST = [0.002, 0.005, 0.01]
PGD_EPS_LIST = [0.005, 0.01]
PGD_ALPHA = 0.0025
PGD_STEPS = 5
LINE_NOISE_AMPLITUDES_50HZ = [0.01, 0.03, 0.05]

for path in [EXTRACT_DIR, OUT_ROOT]:
    Path(path).mkdir(parents=True, exist_ok=True)
for subdir in ['csv', 'figures', 'logs']:
    Path(OUT_ROOT, subdir).mkdir(parents=True, exist_ok=True)

repo_root = Path(REPO_ROOT)
npz_path = Path(NPZ_PATH)
assert repo_root.exists(), f'Repo root missing: {repo_root}'
assert npz_path.exists(), f'NPZ missing: {npz_path}'
assert (repo_root / 'scripts' / 'train_60ep.py').exists(), 'Repo upload looks incomplete.'

sys.path.insert(0, str(repo_root))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)

In [ ]:
# Extract the npz once without loading the full dataset into RAM
extract_dir = Path(EXTRACT_DIR)
x_npy = extract_dir / 'X.npy'

if not x_npy.exists():
    print('Extracting NPZ members to:', extract_dir)
    with zipfile.ZipFile(npz_path, 'r') as zf:
        zf.extractall(extract_dir)
else:
    print('Using previously extracted files from:', extract_dir)

print('Extracted files:')
for p in sorted(extract_dir.iterdir()):
    print(' -', p.name)

In [ ]:
# Memory-map the large X array and load smaller arrays normally
X_mm = np.load(extract_dir / 'X.npy', mmap_mode='r')

y_path = extract_dir / 'y.npy'
Y_path = extract_dir / 'Y.npy'
session_path = extract_dir / 'session.npy'
runs_path = extract_dir / 'runs.npy'

if y_path.exists():
    Y = np.load(y_path, allow_pickle=True)
elif Y_path.exists():
    Y = np.load(Y_path, allow_pickle=True)
else:
    raise FileNotFoundError('No y.npy or Y.npy found after extraction.')

if session_path.exists():
    runs_raw = np.load(session_path, allow_pickle=True)
elif runs_path.exists():
    runs_raw = np.load(runs_path, allow_pickle=True)
else:
    raise FileNotFoundError('No session.npy or runs.npy found after extraction.')

def canon_run(x):
    x = str(x).lower().strip()
    x = x.replace('session', '').replace('_', '').replace('-', '')
    if x.startswith('r') and x[1:].isdigit():
        return 'r' + x[1:].zfill(2)
    if x.startswith('run') and x[3:].isdigit():
        return 'r' + str(int(x[3:])).zfill(2)
    return x

runs = np.array([canon_run(r) for r in runs_raw], dtype=object)

print('X shape:', X_mm.shape)
print('Y shape:', Y.shape)
print('Subjects:', len(np.unique(Y)))
print('Unique runs:', sorted(set(runs.tolist()))[:20])

In [ ]:
# Build B2T indices without copying the full X array
train_idx = np.where(np.isin(runs, TRAIN_RUNS))[0]
test_idx = np.where(np.isin(runs, TEST_RUNS))[0]

Y_train = Y[train_idx]
Y_test = Y[test_idx]
runs_test = runs[test_idx]
state_all = np.array([0 if r in TRAIN_RUNS else 1 for r in runs], dtype=np.int64)
state_train = state_all[train_idx]
state_test = state_all[test_idx]

print('Train windows:', len(train_idx))
print('Test windows :', len(test_idx))

In [ ]:
# Low-RAM smoke-test subset
if RUN_SMOKE_TEST:
    keep_subjects = np.unique(Y_train)[:MAX_SUBJECTS_SMOKE]
    train_keep = np.where(np.isin(Y_train, keep_subjects))[0]
    test_keep = np.where(np.isin(Y_test, keep_subjects))[0][:MAX_PROBES_SMOKE]

    train_idx_eval = train_idx[train_keep]
    test_idx_eval = test_idx[test_keep]
    Y_train_eval = Y[train_idx_eval]
    Y_test_eval = Y[test_idx_eval]
    state_train_eval = state_all[train_idx_eval]
    state_test_eval = state_all[test_idx_eval]
    runs_test_eval = runs[test_idx_eval]
else:
    train_idx_eval = train_idx
    test_idx_eval = test_idx
    Y_train_eval = Y_train
    Y_test_eval = Y_test
    state_train_eval = state_train
    state_test_eval = state_test
    runs_test_eval = runs_test

print('Eval train windows:', len(train_idx_eval))
print('Eval test windows :', len(test_idx_eval))
print('Eval subjects     :', len(np.unique(Y_train_eval)))

In [ ]:
# Checkpoint-compatible model from train_60ep.py
class EEGBackbone(nn.Module):
    def __init__(self, Cin=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(Cin, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.ELU(),
            nn.Conv1d(64, 128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.ELU(),
            nn.Conv1d(128, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ELU(),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

class EEGDisentangle(nn.Module):
    def __init__(self, Cin=64, emb_dim=128):
        super().__init__()
        self.backbone = EEGBackbone(Cin)
        self.id_head = nn.Sequential(nn.Linear(256, emb_dim), nn.LayerNorm(emb_dim))
        self.state_head = nn.Sequential(nn.Linear(emb_dim, 64), nn.ReLU(), nn.Linear(64, 2))
        self.cs_head = nn.Sequential(nn.Linear(256, emb_dim), nn.LayerNorm(emb_dim))

    def forward(self, x):
        f = self.backbone(x)
        z_id = F.normalize(self.id_head(f), dim=1)
        z_cs = F.normalize(self.cs_head(f), dim=1)
        return z_id, z_cs

ckpt_candidates = [
    repo_root / 'checkpoints' / 'seed_1' / 'model_best.pt',
    repo_root / 'checkpoints' / 'seed_1' / 'checkpoint_best.pt',
]
ckpt_path = None
for c in ckpt_candidates:
    if c.exists():
        ckpt_path = c
        break

if ckpt_path is None:
    raise FileNotFoundError('No shipped checkpoint found in the uploaded repo. Please finish the repo upload first.')

model = EEGDisentangle(Cin=64, emb_dim=128).to(DEVICE)
ckpt = torch.load(ckpt_path, map_location=DEVICE)
state = ckpt['model_state'] if 'model_state' in ckpt else ckpt
model.load_state_dict(state)
model.eval()
print('Loaded checkpoint:', ckpt_path)

In [ ]:
# Dataset wrappers that read from memory-mapped X on demand
class IndexedEEGDataset(Dataset):
    def __init__(self, x_mm, indices, labels, states=None):
        self.x_mm = x_mm
        self.indices = np.asarray(indices)
        self.labels = np.asarray(labels)
        self.states = None if states is None else np.asarray(states)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        x = torch.tensor(np.asarray(self.x_mm[idx], dtype=np.float32))
        y = torch.tensor(int(self.labels[i]), dtype=torch.long)
        if self.states is None:
            s = torch.tensor(0, dtype=torch.long)
        else:
            s = torch.tensor(int(self.states[i]), dtype=torch.long)
        return x, y, s

def extract_embeddings_memmap(model, x_mm, indices, labels, states, batch_size=EMBED_BATCH_SIZE):
    ds = IndexedEEGDataset(x_mm, indices, labels, states)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    embs = []
    with torch.no_grad():
        for xb, _, _ in loader:
            xb = xb.to(DEVICE)
            z_id, _ = model(xb)
            embs.append(z_id.cpu().numpy())
    E = np.concatenate(embs, axis=0)
    return E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)

In [ ]:
# Clean B2T evaluation
def build_prototypes(E_train, Y_train, K=3):
    pvecs = []
    powner = []
    for sid in sorted(np.unique(Y_train).tolist()):
        idx = np.where(Y_train == sid)[0]
        Xs = E_train[idx]
        k_use = min(K, len(Xs))
        km = KMeans(n_clusters=k_use, random_state=0, n_init=10).fit(Xs)
        for c in km.cluster_centers_:
            c = c / (np.linalg.norm(c) + 1e-12)
            pvecs.append(c.astype(np.float32))
            powner.append(int(sid))
    return np.stack(pvecs), np.array(powner, dtype=np.int64)

def collect_scores(E_test, Y_test, pvecs, powner, chunk_size=SCORE_CHUNK_SIZE):
    genuine_chunks = []
    impostor_chunks = []
    for start in range(0, len(E_test), chunk_size):
        stop = min(len(E_test), start + chunk_size)
        Eb = E_test[start:stop]
        Yb = Y_test[start:stop]
        sim = Eb @ pvecs.T
        genuine_part = np.empty(len(Eb), dtype=np.float32)
        impostor_part = []
        for i in range(len(Eb)):
            yt = Yb[i]
            mg = powner == yt
            mi = powner != yt
            genuine_part[i] = np.float32(sim[i, mg].max())
            impostor_part.append(sim[i, mi].astype(np.float32))
        genuine_chunks.append(genuine_part)
        impostor_chunks.append(np.concatenate(impostor_part, axis=0))
    return np.concatenate(genuine_chunks, axis=0), np.concatenate(impostor_chunks, axis=0)

def compute_metrics(genuine, impostor, threshold=None):
    scores = np.concatenate([genuine, impostor])
    labels = np.concatenate([np.ones(len(genuine)), np.zeros(len(impostor))]).astype(np.int32)
    fpr, tpr, thrs = roc_curve(labels, scores, pos_label=1)
    auc_val = sk_auc(fpr, tpr)
    fnr = 1.0 - tpr
    idx = int(np.nanargmin(np.abs(fpr - fnr)))
    eer = float((fpr[idx] + fnr[idx]) / 2.0)
    tau = float(thrs[idx]) if threshold is None else float(threshold)
    far = float((impostor >= tau).mean())
    frr = float((genuine < tau).mean())
    return {'auc': float(auc_val), 'eer': eer, 'threshold': tau, 'far': far, 'frr': frr}

E_train = extract_embeddings_memmap(model, X_mm, train_idx_eval, Y_train_eval, state_train_eval)
E_test = extract_embeddings_memmap(model, X_mm, test_idx_eval, Y_test_eval, state_test_eval)
pvecs, powner = build_prototypes(E_train, Y_train_eval, K=K_PROTOTYPES)
genuine_clean, impostor_clean = collect_scores(E_test, Y_test_eval, pvecs, powner)
clean_metrics = compute_metrics(genuine_clean, impostor_clean)
pd.DataFrame([{'condition': 'clean', **clean_metrics}]).to_csv(Path(OUT_ROOT) / 'csv' / 'clean_baseline.csv', index=False)
print(clean_metrics)

In [ ]:
# Attack utilities
proto_map = {int(sid): pvecs[powner == sid] for sid in np.unique(powner)}
all_target_subjects = sorted(list(proto_map.keys()))
if ATTACK_TARGET_SUBJECT_LIMIT is None:
    target_subjects = all_target_subjects
else:
    target_subjects = all_target_subjects[:ATTACK_TARGET_SUBJECT_LIMIT]

def norm_rows_t(x, dim=1):
    return x / (torch.norm(x, dim=dim, keepdim=True) + 1e-12)

def score_vs_subject(model, xb, proto_matrix):
    proto_t = torch.tensor(proto_matrix, dtype=torch.float32, device=DEVICE)
    proto_t = norm_rows_t(proto_t, dim=1)
    z_id, _ = model(xb)
    z_id = norm_rows_t(z_id, dim=1)
    return torch.max(z_id @ proto_t.T, dim=1).values

def embed_array(model, x_np, batch_size=ATTACK_BATCH_SIZE):
    x_t = torch.tensor(x_np, dtype=torch.float32)
    ds = IndexedEEGDataset(x_t, np.arange(len(x_t)), np.zeros(len(x_t), dtype=np.int64), np.zeros(len(x_t), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    embs = []
    with torch.no_grad():
        for xb, _, _ in dl:
            z_id, _ = model(xb.to(DEVICE))
            embs.append(z_id.cpu().numpy())
    E = np.concatenate(embs, axis=0)
    return E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)

def fgsm_attack(model, x_np, proto_matrix, eps):
    x = torch.tensor(x_np, dtype=torch.float32, device=DEVICE, requires_grad=True)
    objective = score_vs_subject(model, x, proto_matrix).mean()
    model.zero_grad(set_to_none=True)
    objective.backward()
    adv = x + eps * x.grad.sign()
    return adv.detach().cpu().numpy().astype(np.float32)

def pgd_attack(model, x_np, proto_matrix, eps, alpha, steps):
    x0 = torch.tensor(x_np, dtype=torch.float32, device=DEVICE)
    x = x0.clone().detach()
    for _ in range(steps):
        x.requires_grad_(True)
        objective = score_vs_subject(model, x, proto_matrix).mean()
        model.zero_grad(set_to_none=True)
        objective.backward()
        x = x.detach() + alpha * x.grad.sign()
        delta = torch.clamp(x - x0, min=-eps, max=eps)
        x = (x0 + delta).detach()
    return x.detach().cpu().numpy().astype(np.float32)

def line_noise_50hz(x_np, amplitude, sfreq=128.0):
    x_out = x_np.astype(np.float32).copy()
    t = np.arange(x_out.shape[-1], dtype=np.float32) / sfreq
    wave = np.sin(2 * np.pi * 50.0 * t)[None, :]
    for i in range(len(x_out)):
        x_out[i] += amplitude * (x_out[i].std() + 1e-8) * wave
    return x_out

In [ ]:
# FGSM / PGD attack sweeps on a manageable impostor subset
def run_attack_grid(mode='fgsm', eps_list=None, alpha=PGD_ALPHA, steps=PGD_STEPS):
    eps_list = eps_list or [0.01]
    rows = []
    threshold = clean_metrics['threshold']
    clean_genuine = genuine_clean.copy()

    for eps in eps_list:
        before = []
        after = []
        success = []

        unique_test_subjects = np.unique(Y_test_eval)
        for target_sid in target_subjects:
            proto = proto_map[target_sid]
            proto_norm = proto / (np.linalg.norm(proto, axis=1, keepdims=True) + 1e-12)
            if ATTACK_IMPOSTOR_LIMIT is None:
                impostor_ids = [sid for sid in unique_test_subjects if sid != target_sid]
            else:
                impostor_ids = [sid for sid in unique_test_subjects if sid != target_sid][:ATTACK_IMPOSTOR_LIMIT]

            for imp_sid in impostor_ids:
                local_idx = np.where(Y_test_eval == imp_sid)[0][:ATTACK_WINDOWS_PER_IMPOSTOR]
                if len(local_idx) == 0:
                    continue
                real_indices = test_idx_eval[local_idx]
                xb = np.asarray(X_mm[real_indices], dtype=np.float32)

                E_before = embed_array(model, xb, batch_size=ATTACK_BATCH_SIZE)
                s_before = (E_before @ proto_norm.T).max(axis=1)

                if mode == 'fgsm':
                    x_adv = fgsm_attack(model, xb, proto, eps)
                else:
                    x_adv = pgd_attack(model, xb, proto, eps, alpha, steps)

                E_after = embed_array(model, x_adv, batch_size=ATTACK_BATCH_SIZE)
                s_after = (E_after @ proto_norm.T).max(axis=1)

                before.extend(s_before.tolist())
                after.extend(s_after.tolist())
                success.extend((s_after >= threshold).astype(np.int32).tolist())

        attacked_impostor = np.array(after, dtype=np.float64)
        attacked_metrics = compute_metrics(clean_genuine, attacked_impostor, threshold=threshold)

        rows.append({
            'attack': mode.upper(),
            'epsilon': float(eps),
            'alpha': float(alpha if mode == 'pgd' else 0.0),
            'steps': int(steps if mode == 'pgd' else 1),
            'threshold': float(threshold),
            'auc': attacked_metrics['auc'],
            'eer': attacked_metrics['eer'],
            'far': attacked_metrics['far'],
            'frr': attacked_metrics['frr'],
            'attack_success_rate': float(np.mean(success) if success else 0.0),
            'mean_impostor_score_before': float(np.mean(before) if before else 0.0),
            'mean_impostor_score_after': float(np.mean(after) if after else 0.0),
            'genuine_score_drop': 0.0,
            'impostor_score_rise': float((np.mean(after) - np.mean(before)) if before else 0.0),
        })
    return pd.DataFrame(rows)

df_fgsm = run_attack_grid(mode='fgsm', eps_list=FGSM_EPS_LIST)
df_pgd = run_attack_grid(mode='pgd', eps_list=PGD_EPS_LIST, alpha=PGD_ALPHA, steps=PGD_STEPS)

df_fgsm.to_csv(Path(OUT_ROOT) / 'csv' / 'fgsm_attack_results.csv', index=False)
df_pgd.to_csv(Path(OUT_ROOT) / 'csv' / 'pgd_attack_results.csv', index=False)
display(df_fgsm)
display(df_pgd)

In [ ]:
# 50 Hz robustness sweep
line_rows = []
for amp in LINE_NOISE_AMPLITUDES_50HZ:
    genuine_parts = []
    impostor_parts = []
    for start in range(0, len(test_idx_eval), NOISE_CHUNK_SIZE):
        stop = min(len(test_idx_eval), start + NOISE_CHUNK_SIZE)
        idx_chunk = test_idx_eval[start:stop]
        y_chunk = Y_test_eval[start:stop]
        x_chunk = np.asarray(X_mm[idx_chunk], dtype=np.float32)
        x_noisy = line_noise_50hz(x_chunk, amp, sfreq=128.0)
        E_chunk = embed_array(model, x_noisy, batch_size=EMBED_BATCH_SIZE)
        genuine_chunk, impostor_chunk = collect_scores(E_chunk, y_chunk, pvecs, powner, chunk_size=len(E_chunk))
        genuine_parts.append(genuine_chunk.astype(np.float32))
        impostor_parts.append(impostor_chunk.astype(np.float32))
    genuine_noisy = np.concatenate(genuine_parts, axis=0)
    impostor_noisy = np.concatenate(impostor_parts, axis=0)
    met = compute_metrics(genuine_noisy, impostor_noisy, threshold=clean_metrics['threshold'])
    met.update({
        'attack': 'LINE_NOISE_50HZ',
        'amplitude': float(amp),
        'genuine_score_drop': float(genuine_clean.mean() - genuine_noisy.mean()),
        'impostor_score_rise': float(impostor_noisy.mean() - impostor_clean.mean()),
    })
    line_rows.append(met)

df_line = pd.DataFrame(line_rows)
df_line.to_csv(Path(OUT_ROOT) / 'csv' / 'line_noise_50hz_results.csv', index=False)
display(df_line)

In [ ]:
# Merged summary and simple figures
summary_rows = [
    {'attack': 'CLEAN', 'setting': 'baseline', **clean_metrics, 'attack_success_rate': np.nan, 'genuine_score_drop': 0.0, 'impostor_score_rise': 0.0}
]

for frame in [df_fgsm, df_pgd, df_line]:
    for _, row in frame.iterrows():
        if row['attack'] == 'LINE_NOISE_50HZ':
            setting = f"amp={row['amplitude']}"
            asr = np.nan
        elif row['attack'] == 'FGSM':
            setting = f"eps={row['epsilon']}"
            asr = row['attack_success_rate']
        else:
            setting = f"eps={row['epsilon']},steps={int(row['steps'])}"
            asr = row['attack_success_rate']

        summary_rows.append({
            'attack': row['attack'],
            'setting': setting,
            'auc': row['auc'],
            'eer': row['eer'],
            'threshold': row['threshold'],
            'far': row['far'],
            'frr': row['frr'],
            'attack_success_rate': asr,
            'genuine_score_drop': row['genuine_score_drop'],
            'impostor_score_rise': row['impostor_score_rise'],
        })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(Path(OUT_ROOT) / 'csv' / 'attack_summary_merged.csv', index=False)
display(df_summary)

plt.figure(figsize=(7, 5))
plt.plot(df_fgsm['epsilon'], df_fgsm['attack_success_rate'], marker='o', label='FGSM ASR')
plt.plot(df_pgd['epsilon'], df_pgd['attack_success_rate'], marker='s', label='PGD ASR')
plt.xlabel('Epsilon')
plt.ylabel('Attack Success Rate')
plt.title('FGSM / PGD Attack Success Rate (Low-RAM Colab)')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUT_ROOT) / 'figures' / 'fgsm_pgd_asr.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(df_fgsm['epsilon'], df_fgsm['eer'], marker='o', label='FGSM EER')
plt.plot(df_pgd['epsilon'], df_pgd['eer'], marker='s', label='PGD EER')
plt.xlabel('Epsilon')
plt.ylabel('EER')
plt.title('Verification EER Under Adversarial Impostor Attacks')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUT_ROOT) / 'figures' / 'fgsm_pgd_eer.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(df_line['amplitude'], df_line['eer'], marker='o', label='EER')
plt.plot(df_line['amplitude'], df_line['auc'], marker='s', label='AUC')
plt.xlabel('50 Hz noise amplitude')
plt.ylabel('Metric value')
plt.title('50 Hz Line-noise Robustness (Low-RAM Colab)')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUT_ROOT) / 'figures' / 'line_noise_50hz_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
manifest = {
    'repo_root': REPO_ROOT,
    'npz_path': NPZ_PATH,
    'extract_dir': EXTRACT_DIR,
    'out_root': OUT_ROOT,
    'checkpoint_path': str(ckpt_path),
    'run_smoke_test': RUN_SMOKE_TEST,
    'max_subjects_smoke': MAX_SUBJECTS_SMOKE,
    'max_probes_smoke': MAX_PROBES_SMOKE,
    'embed_batch_size': EMBED_BATCH_SIZE,
    'score_chunk_size': SCORE_CHUNK_SIZE,
    'noise_chunk_size': NOISE_CHUNK_SIZE,
    'attack_target_subject_limit': ATTACK_TARGET_SUBJECT_LIMIT,
    'attack_impostor_limit': ATTACK_IMPOSTOR_LIMIT,
    'attack_windows_per_impostor': ATTACK_WINDOWS_PER_IMPOSTOR,
    'clean_metrics': clean_metrics,
}
with open(Path(OUT_ROOT) / 'logs' / 'run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('Low-RAM notebook finished.')
print('Outputs saved to:', OUT_ROOT)